# OcuWeave Native
### Reproducible segmentation infrastructure and disease classification
Marchel Andrian Shevchenko · Data Sorcerers · 2026

Release **v0.13.0**. Native C++/CUDA operators, analytic gradients and AdamW; random initialization. Python orchestrates preparation and reporting. [Repository](https://github.com/chenko-the-sorcerers/ocuweave) · [Protocol](https://github.com/chenko-the-sorcerers/ocuweave/blob/v0.13.0/docs/protocols/native/classification.md)

**Execution status:** this Kaggle launcher has not executed medical training. GPU and internet require Kaggle account verification. CPU tests are reported separately in the release. Validation Dice >80% and classification accuracy >85% are objectives, not results.

This launcher tests both native segmentation and classification, then optionally trains the registered FIVES classifier. Historical Colab segmentation runs require a separate checkpoint review and are not silently restarted here.

## 1. Hardware and workspace
Enable a GPU and internet in Kaggle settings after account verification. No purchase is required by this notebook. Output files go to `/kaggle/working`; save a notebook version and preserve outputs before ending the session. Google Drive is not mounted automatically.

In [ ]:
from pathlib import Path
import datetime, json, platform, subprocess, sys
from PIL import __version__ as pillow_version

WORK = Path('/kaggle/working')
if not WORK.is_dir():
    raise RuntimeError('This launcher expects a Kaggle runtime')
REPO = WORK / 'ocuweave-source'
BUILD = WORK / 'ocuweave-native-build'
RELEASE = 'v0.13.0'
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True, check=True)
print(GPU.stdout)
capability = GPU.stdout.splitlines()[0].split(',')[1].strip()
CUDA_ARCH = capability.replace('.', '')
if not CUDA_ARCH.isdigit():
    raise RuntimeError('Cannot determine allocated GPU compute capability')
print('CUDA architecture:', CUDA_ARCH, '| Pillow:', pillow_version)

## 2. Release source and native numerical tests
The release tag is resolved to a commit and recorded. An existing checkout must match the release and be clean. Compilation and CTest must succeed before training; CUDA tests compare native GPU outputs and gradients against the CPU implementation.

In [ ]:
def run(args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if not REPO.exists():
    run(['git', 'clone', '--branch', RELEASE, '--depth', '1',
         'https://github.com/chenko-the-sorcerers/ocuweave.git', REPO])
def git(*args):
    return subprocess.check_output(['git', *args], cwd=REPO, text=True).strip()
SOURCE_COMMIT = git('rev-parse', 'HEAD')
if SOURCE_COMMIT != git('rev-parse', RELEASE + '^{commit}') or git('status', '--porcelain'):
    raise RuntimeError('Existing source does not match the clean release')
run(['cmake', '-S', REPO, '-B', BUILD, '-DCMAKE_BUILD_TYPE=Release',
     '-DOCUWEAVE_ENABLE_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=' + CUDA_ARCH])
run(['cmake', '--build', BUILD, '-j2'])
run(['ctest', '--test-dir', BUILD, '--output-on-failure'])
print('Native numerical checks passed; source:', SOURCE_COMMIT)

## 3. Frozen inputs and training gate
Attach the original FIVES data and the existing frozen development manifest as **private Kaggle inputs**. Preserve dataset license/attribution. Set the extraction root so each manifest image path resolves under it. Do not regenerate the split or use test labels. No vessel mask is used for classification.

The required manifest SHA-256 is `632b76bad18375cb824fc18761216a00428310fe806f4170c2f866199f13ad68`.

This baseline uses AMD, diabetic retinopathy, glaucoma and normal labels; seed 42; 192 px; width 24; batch 8; 1,200 steps; no augmentation. It does not demonstrate early detection or predict vision loss. Patient identity is still unverified. Set `RUN_TRAINING = True` only once the exact inputs are attached.

In [ ]:
DATA_ROOT = None  # e.g. Path('/kaggle/input/your-private-input/FIVES')
MANIFEST = None   # Path to the exact frozen manifest.json
RUN_TRAINING = False

if RUN_TRAINING:
    if DATA_ROOT is None or MANIFEST is None:
        raise ValueError('Set DATA_ROOT and MANIFEST; do not guess data provenance')
    DATA_ROOT, MANIFEST = Path(DATA_ROOT), Path(MANIFEST)
    if not DATA_ROOT.is_dir() or not MANIFEST.is_file():
        raise FileNotFoundError('Frozen data inputs are not attached')
else:
    print('Training is disabled. Attach verified inputs before enabling it.')

## 4. Train and independently recount validation
The runner enforces the frozen manifest hash, verifies image bytes, excludes test data, checks native code and creates a new immutable output session. A classification run may use this separate GPU while earlier segmentation jobs use another machine. This does not modify their configurations or checkpoints.

The final checkpoint is evaluated once at the registered budget. Accuracy, balanced accuracy, macro F1, class-specific sensitivity/specificity and AUROC are recounted from saved probabilities. A high training score is not a validation result.

In [ ]:
if RUN_TRAINING:
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    SESSION = WORK / 'ocuweave-runs' / ('fives-classification-' + stamp)
    environment = {'platform': 'Kaggle', 'gpu': GPU.stdout.strip(),
                   'python': sys.version, 'system': platform.platform(),
                   'pillow': pillow_version, 'release': RELEASE,
                   'source_commit': SOURCE_COMMIT, 'started_utc': stamp}
    # The programme refuses an existing session. Write environment after its creation.
    process = subprocess.Popen([
        sys.executable, str(REPO / 'scripts/native_classification_programme.py'),
        '--repo', str(REPO), '--build', str(BUILD), '--root', str(DATA_ROOT),
        '--manifest', str(MANIFEST), '--session', str(SESSION),
        '--expected-commit', SOURCE_COMMIT], cwd=REPO)
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        print('Interrupted: inspect the process and latest checkpoint before resuming.')
        raise
    finally:
        if SESSION.is_dir():
            (SESSION / 'kaggle-environment.json').write_text(json.dumps(environment, indent=2) + '\n')
    if return_code:
        raise RuntimeError('Programme failed; inspect ' + str(SESSION / 'status.json'))
    print((SESSION / 'status.json').read_text())
else:
    print('No medical-data training started.')

## 5. Preserve evidence
Save a Kaggle version with outputs, then transfer the entire run directory to the OcuWeave Drive runs folder. Preserve contracts, environment, checksums, predictions, logs, evaluation and checkpoints. A draft session alone is not archival storage.

Do not publish data or weights without their corresponding license and model card. Record failures as well as successful runs. Existing Colab segmentation checkpoints must be inspected for the last successfully saved optimizer step before a separate migration. The low-level native classifier supports checkpoint resume, but the complete source/data/configuration must remain identical; this notebook creates a fresh classification session.

**Reference:** Y. Jin et al., “FIVES: A Fundus Image Dataset for AI-based Vessel Segmentation,” *Scientific Data*, vol. 9, 2022. [Paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC9352679/).